#### Cross‑entropy loss

Cross-entropy measures the distance between these two distributions.
- The Intuition: If the actual next word is "apple", and your model predicted "apple" with 99% probability, your loss is very low. If it predicted "apple" with 1% probability, your loss is very high.
- The Math: For a single word prediction, cross-entropy loss simplifies to the negative log probability of the correct word.
$$L_{CE} = - \log P(w_{target})$$

#### Perplexity (PPL)
Perplexity is the standard evaluation metric for language models. It is directly derived from cross-entropy.
- The Intuition: Think of perplexity as the model's "branching factor" or its level of surprise. A perplexity of 10 means that, on average, the model is as confused as if it had to guess the next word from a list of 10 equally likely options. Lower is always better.
- The Math: Perplexity is the exponentiated average cross-entropy loss. For a sequence of $N$ words:
$$PPL = 2^{-\frac{1}{N} \sum_{i=1}^{N} \log_2 P(w_i | w_1...w_{i-1})}$$
If you are using the natural logarithm ($e$), then $PPL = e^{Loss}$.

| Feature | N-Gram Language Model | Word2Vec Prediction Task |
|---|---|---|
| Primary Output | Text generation and sequence probabilities | Dense word vectors (embeddings) |
| Word Representation | Discrete symbols (exact string matches) | Continuous mathematical vectors |
| Handling Unseen Data | Fails or requires complex smoothing | Generalizes well via semantic similarity |
| Context Direction | Left-to-right (past only) | Usually bidirectional (surrounding window) |

The N-gram model, which is great at predicting sequences but bad at understanding meaning. And There is Word2Vec, which is great at understanding meaning but isn't built to generate long sequences of text.

In [ ]:
import random
from collections import defaultdict, Counter

class SimpleNgramLanguageModel:
    def __init__(self, n):
        self.n = n
        self.ngrams = defaultdict(Counter)
        self.unigram_counts = Counter()
    
    def train(self, text, n):
        for sentence in text:
            tokens = ['<s>'] + sentence.lower().split() + ['</s>']

        for i in range(len(tokens) - n + 1):
            ngrams = tuple(tokens[i:i+n])
            self.ngrams[ngrams[:-1]][ngrams[-1]] += 1
            self.unigram_counts[ngrams[-1]] += 1
    


    def generate_text(self, start_ngram=('<s>',), length=10):
        current_ngram = start_ngram
        generated_text = []

        for _ in range(length):
            if current_ngram not in self.ngrams:
                break

            next_words = list(self.ngrams[current_ngram].keys())
            next_word_weights = list(self.ngrams[current_ngram].values())

            next_word = random.choices(next_words, weights=next_word_weights)[0]

            if next_word == '</s>':
                break

            generated_text.append(next_word)

            # slide window forward
            if isinstance(current_ngram, tuple):
                current_ngram = current_ngram[1:] + (next_word,)
            else:
                current_ngram = next_word

        return ' '.join(generated_text)

# --- Let's test it! ---
training_data = [
    "I love reading books",
    "I love eating pizza",
    "reading books is fun",
    "eating pizza is great"
]

model = SimpleNgramLanguageModel(n=2)
model.train(training_data, n=2)

print(f"Generated text: '{model.generate_text()}'")

Generated text: 'eating pizza is great'


#### The Problem: The Tyranny of Zero
- Imagine we use your bigram model, trained only on your four sentences, to evaluate the probability of a new sentence: "I love eating apples."
The model calculates the probability step-by-step. When it gets to the transition from "eating" to "apples", Using standard probability, the math is:$$P(\text{apples} \mid \text{eating}) = \frac{\text{Count}(\text{eating apples})}{\text{Count}(\text{eating})} = \frac{0}{2} = 0$$
- Because language models calculate the probability of a whole sentence by multiplying the probabilities of each word together, a single zero ruins everything. The entire sentence gets a probability of exactly 0. In log space, taking the $\log(0)$ results in negative infinity, meaning your model's perplexity explodes.
#### Laplace (Add-One) Smoothing
- Laplace smoothing is a clever, simple mathematical trick: we pretend we have seen every possible word transition at least one extra time.

- Instead of starting our counts at zero, we start them all at one. This takes a tiny sliver of probability mass away from the frequent events we have seen and redistributes it to the unseen events, ensuring nothing ever has a probability of exactly zero.

- Standard Probability:$$P(w_i \mid w_{i-1}) = \frac{c}{N}$$
- Laplace Smoothed Probability:$$P(w_i \mid w_{i-1}) = \frac{c + 1}{N + V}$$Notice that we add $1$ to the numerator, but we must add $V$ to the denominator. We do this to ensure that if we sum up the probabilities of all possible next words, they still perfectly add up to 1.0.

- In practice, adding a whole $1$ to every unseen event is often too aggressive, especially when your vocabulary $V$ is massive (e.g., 50,000 words). It steals too much probability away from the actual patterns the model learned.To fix this, we use a fractional value, usually denoted as $\alpha$ (alpha), which might be 0.1, 0.01, or even smaller.$$P(w_i \mid w_{i-1}) = \frac{c + \alpha}{N + \alpha V}$$
- This provides just enough probability to avoid a mathematical crash, without distorting the model's true learned distributions.

In [12]:
import math
import re
import random
from collections import Counter

class NgramLanguageModel:
    def __init__(self, n=2, smoothing=1):
        self.n = n
        self.smoothing = smoothing
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()
        self.vocab_size = 0

    def tokenize(self, text):
        return re.findall(r'\b\w+\b', text.lower())

    def add_sentence_tokens(self, tokens):
        return ['<s>'] * (self.n - 1) + tokens + ['</s>']

    def train(self, text):
        for sentence in text:
            tokens = self.tokenize(sentence)
            tokens = self.add_sentence_tokens(tokens)
            self.vocab.update(tokens)

            for i in range(len(tokens) - self.n + 1):
                ngram = tuple(tokens[i:i+self.n])
                context = ngram[:-1]

                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

        self.vocab_size = len(self.vocab)

    def probability(self, context, word):
        context = tuple(context)

        if len(context) != self.n - 1:
            context = context[-(self.n - 1):]

        ngram = context + (word,)
        ngram_count = self.ngram_counts[ngram]
        context_count = self.context_counts[context]

        return (ngram_count + self.smoothing) / (
            context_count + self.smoothing * self.vocab_size
        )

    def next_word_probabilities(self, context):
        context = tuple(context)

        if len(context) != self.n - 1:
            context = context[-(self.n - 1):]

        probs = {}
        for word in self.vocab:
            probs[word] = self.probability(context, word)
        return probs

    def predict_next(self, context):
        probs = self.next_word_probabilities(context)
        return max(probs, key=probs.get)

    def generate_text(self, start_words=None, length=10):
        if start_words is None:
            context = ['<s>'] * (self.n - 1)
        else:
            if isinstance(start_words, str):
                start_words = self.tokenize(start_words)

            context = ['<s>'] * max(0, self.n - 1 - len(start_words)) + start_words
            context = context[-(self.n - 1):]

        generated = []

        for _ in range(length):
            probs = self.next_word_probabilities(context)
            words = list(probs.keys())
            weights = list(probs.values())

            next_word = random.choices(words, weights=weights)[0]

            if next_word == '</s>':
                break

            generated.append(next_word)

            if self.n > 1:
                context = context[1:] + [next_word]

        return ' '.join(generated)

    def perplexity(self, text):
        tokens = self.tokenize(text)
        tokens = self.add_sentence_tokens(tokens)

        log_prob_sum = 0.0
        N = len(tokens) - self.n + 1

        for i in range(N):
            ngram = tuple(tokens[i:i+self.n])
            context = ngram[:-1]
            word = ngram[-1]

            prob = self.probability(context, word)
            log_prob_sum += math.log(prob)

        return math.exp(-log_prob_sum / N)

In [ ]:
# # Load and slice to 500k tokens (fast enough on Mac MPS)
# with open("text8") as f:
#     text = f.read()

# words = text.split()[:500000]   # slice here — can be increased later if you want

# chunk_size = 20
# corpus = [" ".join(words[i:i+chunk_size])
#           for i in range(0, len(words), chunk_size)]

In [29]:
filename = "shakespeare.txt"

with open(filename, 'r', encoding='utf-8') as f:
    raw_text = f.read()

In [39]:
# Example usage
# corpus = [
#     "I love machine learning",
#     "I love deep learning",
#     "I enjoy natural language processing",
#     "I love natural language processing.",
#     "I love machine learning.",
#     "Language models predict the next word."
# ]



lm = NgramLanguageModel(n=1, smoothing=1)
lm.train(raw_text.splitlines())

print(lm.predict_next(["i"]))
print(lm.generate_text(start_words="i", length=5))
print(lm.perplexity("i"))

earldom
oddest sluggard hit calamity bloodless
18.18112638729732
